In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/NGC-3217_2_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/NGC-3217_2_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     359   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        40   (2094, 964)   float32   


In [3]:
image_cut = image[67:167, 0:1890]
image_error_cut = image_error[67:167, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.03191
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)# +
        #sersic_1d(x_hr, params5) #+
        #ersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/PSF/Real_seeing_NGC3217_2.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    #choco_area = single_integral(x, params5, a, b)
    #purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area# + choco_area# + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area]#, (choco_area*100)/sum_area]#, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]

## OIII

In [8]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/OIII_fit.csv", index_col = 0)

In [9]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 1'].iloc[3] -1, fit_OIII['Component 1'].iloc[3] + 1)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 2'].iloc[3] -1, fit_OIII['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 3'].iloc[3] -1, fit_OIII['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 4'].iloc[3] -1, fit_OIII['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.617362192614562
0.624852109823243
2.455587712513079
2.516565120017778
0.8912086315579604
0.891075557001535
0.4039523124728221
0.4058343020065493
            Peak 1     Peak 2     Peak 3     Peak 4
Blue     91.969726   5.171839   8.647416   9.975104
Green     5.345018  90.788906  20.282378   1.846266
Magenta   0.015057   1.764264  58.436123   1.236926
Orange    2.670199   2.274992  12.634084  86.941705


In [10]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.2234252218271304
1.2324631907569927
4.869106102260372
4.947196213654572
1.8512947717844432
1.8460285460643417
0.8118153927204634
0.8142418094778034
            Peak 1     Peak 2     Peak 3     Peak 4
Blue     92.717013   5.430276   8.605498  10.200456
Green     4.695002  90.953456  23.202043   2.102406
Magenta   0.012039   1.429534  56.634506   1.570697
Orange    2.575945   2.186735  11.557953  86.126440


In [11]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.2809396156773984
2.2950474626316697
8.525958357754105
8.645293726954593
3.507288087972055
3.504531226527533
1.5673281241470618
1.5710954463462874
            Peak 1     Peak 2     Peak 3     Peak 4
Blue     91.955584   6.231224   9.111276  10.586563
Green     5.254729  89.456625  26.082441   2.258598
Magenta   0.014461   1.801920  52.533119   1.806278
Orange    2.775226   2.510232  12.273163  85.348562


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit_OIII['Component 1'].iloc[3] -1, fit_OIII['Component 1'].iloc[3] + 1)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit_OIII['Component 2'].iloc[3] -1, fit_OIII['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit_OIII['Component 3'].iloc[3] -1, fit_OIII['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit_OIII['Component 4'].iloc[3] -1, fit_OIII['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.6600543278746017
0.6916321857161551
2.9475701414929
3.1115280108190606
1.026113384902261
1.0289509988609251
0.37447443948371006
0.3701723138794861
               Peak 1     Peak 2     Peak 3     Peak 4
Blue     9.018509e+01   3.243713   5.556042   8.075800
Green    7.647273e+00  93.990811  24.834124   3.953338
Magenta  1.380053e-07   1.298518  61.332211   0.023792
Orange   2.167637e+00   1.466958   8.277623  87.947070


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.2923466014746912
1.343369083053607
5.685614693450138
5.951096093373078
2.200592920855913
2.2060249561987595
0.7215982758520472
0.7159408626132091
               Peak 1     Peak 2     Peak 3     Peak 4
Blue     9.096200e+01   3.510982   5.357262   8.601615
Green    6.909932e+00  94.126053  27.182471   4.610848
Magenta  6.707384e-08   0.909032  60.121484   0.050301
Orange   2.128068e+00   1.453933   7.338783  86.737235


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                     fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Magenta', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.362503143353881
2.4325396421560095
10.051299120838197
10.412686354542275
4.241392217297293
4.250360758438057
1.4017125839602227
1.3901720582067703
               Peak 1     Peak 2     Peak 3     Peak 4
Blue     8.984388e+01   3.994701   5.576922   8.872301
Green    7.819780e+00  92.987967  29.866759   4.882996
Magenta  2.300591e-07   1.364719  56.893468   0.094122
Orange   2.336341e+00   1.652613   7.662852  86.150580
